# prenatalppkt walkthrough

This notebook walks through **one real prenatal ultrasound exam** end to end: from the raw Observer JSON export, through every parsing stage, to a validated GA4GH Phenopacket v2 with an attached VCF. The goal is to make the whole pipeline auditable in one sitting - each section runs real code against a real fixture and points at the test file that pins down the behavior being shown.

**Pipeline shape:**

```
Observer JSON / ViewPoint HL7
  -> extractors      (raw fields -> TermBin objects, one per biometry measurement)
  -> section parsers (impression text, anatomy, dating, EFW, ratios -> HPO terms + structured data)
  -> builders         (stitches extractors + section parsers into a Phenopacket, one per fetus)
  -> genomics          (optional: attach a VCF as a File + Interpretation)
```

**Test case:** `tests/data/Apple_Sally_pretty.json` - a singleton pregnancy with a genuinely abnormal finding (a Dandy-Walker malformation), a mix of normal and missing-percentile biometry measurements, and clinical impression text with real negation language ("no evidence of macrocephaly, ventriculomegaly..."). One fixture exercises the normal path, the missing-percentile path, and the abnormal-finding path at once.

This notebook was written ahead of a 2026-07-21 review meeting, and one section (7) documents a real bug this walkthrough surfaced and the fix that followed - included deliberately as an example of the pipeline being audited, not polished after the fact.

## 1. The test case

`Apple_Sally_pretty.json` is a real Observer export shape (field names, nesting, and units match production Observer data), with patient-identifying values replaced by placeholders. Before running any parsing code, look at the raw structure so the rest of the notebook isn't working with a black box.

In [ ]:
import json
from pathlib import Path

data_path = Path("tests/data/Apple_Sally_pretty.json")
with open(data_path) as f:
    observer_data = json.load(f)

print("Top-level keys:", list(observer_data.keys()))
print("Fetuses:", len(observer_data["fetuses"]))
print("Measurements for fetus 1:", [m["label"] for m in observer_data["fetuses"][0]["measurements"]])

## 2. Biometry -> TermBins

Each biometry measurement (HC, BPD, AC, Femur, etc.) is compared against a growth-reference table (`data/mappings/biometry_hpo_mappings.yaml`, loaded by `mapping_loader.py`): 7 measurement types, 8 percentile bins each (≤3, ≤5, ≤10, normal-lower, normal-upper, ≤90, ≤95, ≤97, >97). `PercentileRange.evaluate()` classifies a raw percentile into one of those 8 bins, and each bin is pre-mapped to an HPO term - a normal-range HC maps to nothing abnormal, a >97th-percentile HC maps to "Macrocephaly", etc.

`observer.extract()` runs this for every measurement on a fetus and returns one `TermBin` per measurement. Below, run it on Apple Sally's real data, then build one `TermBin` by hand via `TermBinFactory.create_term_bin()` to see the same mechanism directly, outside the extractor.

In [ ]:
from prenatalppkt.etl.extractors import observer
from prenatalppkt.etl.term_bin_factory import TermBinFactory

term_bins = observer.extract(observer_data)
print(f"Extracted {len(term_bins)} TermBins from fetus 1's measurements:\n")
for tb in term_bins:
    status = "normal" if tb.normal else "ABNORMAL"
    print(f"  {tb.description:45s} [{status:8s}] -> {tb.hpo_id} {tb.hpo_label}")

# Same mechanism, called directly: a 250mm HC at the 42.5th percentile
factory = TermBinFactory()
manual_bin = factory.create_term_bin(
    name="HC", value_mm=250.0, percentile=42.5, gestational_age=None, method="Hadlock"
)
print(f"\nManually-built TermBin: {manual_bin.description} -> {manual_bin.hpo_id} {manual_bin.hpo_label}")

**What proves this works:** four test files, each pinning down one layer of the mechanism above.

| Test file | What it protects |
|---|---|
| `tests/test_mapping_loader.py` | The YAML growth-reference table loads and parses into 8 bins per measurement type |
| `tests/test_reference_range.py` | `PercentileRange.evaluate()` classifies a percentile into the correct one of 8 bins, including the boundary values |
| `tests/etl/test_term_bin_factory.py` | `TermBinFactory.create_term_bin()` maps a (measurement, value, percentile) triple to the correct `TermBin`, for every measurement type |
| `tests/test_term_bin.py` | The `TermBin` dataclass itself - field defaults, `normal` flag semantics |

Run them live below rather than just asserting they pass.

In [ ]:
import pytest

pytest.main([
    "-q",
    "tests/test_mapping_loader.py",
    "tests/test_reference_range.py",
    "tests/etl/test_term_bin_factory.py",
    "tests/test_term_bin.py",
])

## 3. Clinical impression -> SimpleTerms (fenominal)

Biometry maps to HPO through a lookup table because the input is already structured (a labeled numeric value). Free text - the radiologist's impression narrative - needs actual text mining. That's `fenominal` (`hpo/fenominal_cr.py`'s `FenominalConceptRecognizer`): a named-entity-recognition model that finds HPO concept mentions in a string and, critically, detects **negation** ("no evidence of X" -> `SimpleTerm(hpo_id=X, excluded=True)` rather than silently dropping it or marking it present).

`HpoParser` (`hpo/hpo_parser.py`) loads the HPO ontology via `hpotk` and exposes the concept recognizer. Apple Sally's real impression text is worth reading before running it through the recognizer - it names a real finding (Dandy-Walker malformation) and then explicitly rules out three related findings in the same sentence.

In [ ]:
import gzip
from prenatalppkt.hpo import HpoParser
from prenatalppkt.etl.sections import parse_clinical_impression

HP_JSON_GZ = Path("tests/data/hp.json.gz")
TMP_HP_JSON = Path("/tmp/hp_walkthrough.json")
with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
    with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
        f_out.write(f_in.read())

hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
hpo_cr = hpo_parser.get_hpo_concept_recognizer()
print(f"HPO version: {hpo_parser.get_version()}")

impression = parse_clinical_impression(observer_data, "observer_json", hpo_cr=hpo_cr)
print("\nImpression text:\n", impression["impression_text"])
print("\nHPO terms found:")
for term in impression["hpo_terms"]:
    print(f"  {term.hpo_id} {term.hpo_label:30s} excluded={term.excluded}")

**Read the output carefully.** The text says "no evidence of macrocephaly, ventriculomegaly or agenesis of the corpus callosum" - one negated list of three findings. fenominal correctly marks Macrocephaly and Ventriculomegaly `excluded=True`, but marks Agenesis of corpus callosum `excluded=False` even though it's the same negated clause. This is a real, current limitation in fenominal's negation scope detection (it doesn't propagate across an "X, Y or Z" list past the first couple of items) - not a bug in this codebase, and not something to silently work around. Flagging it here because a downstream consumer trusting `excluded` at face value would get one wrong answer out of three in this exact sentence.

**What proves the rest of this works:** `tests/hpo/test_fenominal_cr.py` (the concept recognizer's negation and span-matching logic), `tests/hpo/test_hpo_parser.py` (ontology loading + version reporting), `tests/etl/sections/test_clinical_impression.py` (the section parser wiring above).

## 4. Fetal anatomy -> structured findings + HPO

`parse_fetal_anatomy` classifies each anatomy structure comment (normal / abnormal / not visualized) and runs the free-text anatomy narrative through the same fenominal recognizer to pull out HPO terms for genuine anomalies. Apple Sally's anatomy narrative reports the cerebellum as abnormal ("splaying of the cerebellar hemispheres... consistent with a Dandy-Walker malformation") and, later in the same narrative, explicitly rules out a neural tube defect ("...without evidence of a neural tube defect").

**This second sentence is where a real bug lived until this notebook's own review surfaced it.** fenominal correctly parses that sentence and sets `SimpleTerm(hpo_id="HP:0045005", excluded=True)` - but the builder that turns `SimpleTerm`s into `PhenotypicFeature`s (`_narrative_feature()` in both `observer_phenopacket.py` and `viewpoint_phenopacket.py`) never read `term.excluded`, so every negated anatomy/impression finding silently came out as "observed/present" in the final Phenopacket - the exact opposite of what the source text said. Fixed by adding `excluded=term.excluded` to both builders, with a regression test per builder (`tests/builders/test_observer_phenopacket.py::test_negated_narrative_finding_marked_excluded`, and the ViewPoint equivalent) verified to fail before the fix and pass after. The cell below runs post-fix and shows `excluded=True` for the neural tube defect term.

In [ ]:
from prenatalppkt.etl.sections import parse_fetal_anatomy

anatomy = parse_fetal_anatomy(observer_data, "observer_json", hpo_cr=hpo_cr)
print("Normal structures:", anatomy["normal_structures"])
print("Abnormal structures:", anatomy["abnormal_structures"])
print("Not visualized:", anatomy["not_visualized"])
print("\nHPO terms from the anatomy narrative:")
for term in anatomy["hpo_terms"]:
    print(f"  {term.hpo_id} {term.hpo_label:25s} excluded={term.excluded}")

**A second real bug lived here too, also surfaced by writing this walkthrough.** `_parse_observer_anatomy` read the structured 16-item anatomy array from `fetuses[i]["fetus"]["anatomy"]`, but every real Observer export puts that array at `fetuses[i]["anatomy"]` - a sibling of `"fetus"`, not nested inside it. `normal_structures` / `abnormal_structures` / `not_visualized` / structured `anomalies` came back empty on every real export as a result; the existing unit tests didn't catch it because they hand-built dicts using the same wrong (nested) shape the code expected, so tests and code agreed with each other while both disagreed with reality. Fixed by reading from `fetuses[i]["anatomy"]` directly, plus a related quirk found while re-testing: some real anatomy items set `"detail"`/`"anomalies"` to an explicit `null` rather than omitting the key, which `.get(key, [])` doesn't guard against. The cell above runs post-fix - `Head` and `Cerebellum` now correctly appear in `abnormal_structures`, and the Dandy Walker anomaly appears in `anomalies`, matching the narrative text.

**What proves the rest of this works:** `tests/etl/sections/test_fetal_anatomy.py` (now includes a regression test against this exact fixture), `tests/dto/observer/builders/test_fetus_anatomy_data.py`, `tests/parser/observer/fetuses/test_fetus_anatomy_text_parser.py`.

## 5. The other sections

Four lighter section parsers round out the exam-level context: clinical indication (why the exam was ordered), pregnancy dating (LMP/EDD/GA), estimated fetal weight with SGA/AGA/LGA classification, and biometric ratios (proportionality). Each is a straightforward input -> structured-output transform with its own test file.

In [ ]:
from prenatalppkt.etl.sections import (
    parse_clinical_indication,
    parse_pregnancy_dating,
    parse_estimated_fetal_weight,
    parse_fetal_ratios,
)

indication = parse_clinical_indication(observer_data, "observer_json")
print("Indication:", indication.get("indication_text") or "(not present in this fixture)")

dating = parse_pregnancy_dating(observer_data, "observer_json")
print(f"Dating: lmp={dating.get('lmp')} edd={dating.get('edd')} ga_weeks={dating.get('ga_weeks')}")

efw = parse_estimated_fetal_weight(observer_data, "observer_json")
print(f"EFW: {efw.get('efw_grams')}g at {efw.get('percentile')}th percentile, growth={efw.get('growth_category')}")

ratios = parse_fetal_ratios(observer_data, "observer_json")
print("Ratios:")
for r in ratios["ratios"]:
    print(f"  {r['name']}: {r['value']} (expected {r['expected_range']}) within_range={r['within_range']}")
print("Proportionality:", ratios["proportionality_assessment"])

**What proves this works:** `tests/etl/sections/test_clinical_indication.py`, `tests/etl/sections/test_pregnancy_dating.py`, `tests/etl/sections/test_estimated_fetal_weight.py`, `tests/etl/sections/test_fetal_ratios.py`. Note `ga_weeks` comes back `None` here - `_resolve_subject_ga` in both builders falls back to parsing gestational age out of each measurement's own description text instead (see Section 6 below), a real quirk documented as a TODO in both builder files rather than silently relied upon.

## 6. The builder, end to end

Everything above - biometry, clinical impression, fetal anatomy, dating - is what `build_observer_phenopacket()` stitches together internally. This is the "comprehensive and correct" usage pattern: one call produces a validated `Phenopacket` per fetus, with both bugs from Section 4 (negation not respected, and the anatomy structured-list path) now fixed and covered by regression tests (`tests/builders/test_observer_phenopacket.py::test_negated_narrative_finding_marked_excluded`, `tests/etl/sections/test_fetal_anatomy.py::test_real_fixture_structured_anatomy_is_populated`).

In [ ]:
from datetime import datetime, timezone
from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.builders import build_observer_phenopacket

now_ts = Timestamp()
now_ts.FromDatetime(datetime.now(tz=timezone.utc))

pps = build_observer_phenopacket(observer_data, hpo_parser, now_ts, accession_id="apple-sally")
print(f"Phenopacket count: {len(pps)}")
pp = pps[0]
print(f"id: {pp.id} | subject: {pp.subject.id} | {len(pp.phenotypic_features)} phenotypic features\n")
for pf in pp.phenotypic_features:
    print(f"  {pf.type.id} {pf.type.label:35s} excluded={pf.excluded!s:5} {pf.description[:55]}")

# Round-trip through JSON to confirm the Phenopacket is schema-valid
json_str = MessageToJson(pp)
round_tripped = Parse(json_str, pps2.Phenopacket())
assert round_tripped == pp
print("\nRound-trip through JSON validated OK")

**What proves the builder itself works, beyond this one fixture:** `tests/builders/test_observer_phenopacket.py` covers twin pregnancies (each fetus gets its own Phenopacket with stable subject IDs across exams in the same pregnancy), HPO-id deduplication (HC and BPD both mapping to the same abnormal-skull-size term collapses to one feature), and an UNKNOWN-scan-type fetus producing an empty-but-valid Phenopacket rather than an error. Run the twin and dedup cases live below.

In [ ]:
import pytest

pytest.main([
    "-v",
    "tests/builders/test_observer_phenopacket.py::test_twin_returns_two_phenopackets",
    "tests/builders/test_observer_phenopacket.py::test_hpo_id_dedup_keeps_first_occurrence",
    "tests/builders/test_observer_phenopacket.py::test_unknown_fetus_returns_empty_features_phenopacket",
    "tests/builders/test_observer_phenopacket.py::test_negated_narrative_finding_marked_excluded",
])

## 7. Genomics attachment

A Phenopacket can carry more than phenotype - `files` and `interpretations` let genomic findings travel with the clinical picture. `scan_vcf_file` reads variant loci from a VCF, and `build_vcf_file_entry`/`build_genomic_interpretation` attach them to the Phenopacket built above. **This is structural scaffolding only** - no VRS normalization, no ACMG classification, no variant-to-phenotype linkage logic. It proves the container can hold genomic data; it doesn't yet do anything clinically meaningful with it.

**What proves this works:** `tests/genomics/test_vcf.py`, `tests/genomics/test_genomic.py`.

In [ ]:
from pathlib import Path
from prenatalppkt.genomics import (
    build_genomic_interpretation,
    build_vcf_file_entry,
    scan_vcf_file,
)

vcf_path = Path("tests/data/Apple_Sally.vcf")
variants = scan_vcf_file(vcf_path)
print(f"Scanned {len(variants)} variant loci from {vcf_path.name}:")
for v in variants:
    print(f"  {v.chrom}:{v.pos} {v.ref}>{v.alt} ({v.genome_assembly})")

pp.files.append(
    build_vcf_file_entry(
        vcf_path.resolve().as_uri(),
        attributes={"genomeAssembly": variants[0].genome_assembly},
    )
)
pp.interpretations.append(
    build_genomic_interpretation(
        variants, subject_id=pp.subject.id, interpretation_id=f"{pp.id}-genomic-interp-1"
    )
)
print(f"\nPhenopacket '{pp.id}' now has {len(pp.files)} file(s), {len(pp.interpretations)} interpretation(s)")

## 8. Architecture map

```
src/prenatalppkt/
  etl/
    extractors/     Observer JSON + ViewPoint HL7 -> List[TermBin] (biometry)
    sections/       impression, anatomy, dating, EFW, ratios, indication -> Dicts
    term_bin_factory.py, mapping_loader.py   the growth-reference lookup underneath extractors
  hpo/
    hpo_parser.py     loads the HPO ontology (hpotk)
    fenominal_cr.py   text -> HPO concept recognition, with negation detection
  builders/
    observer_phenopacket.py, viewpoint_phenopacket.py   stitch extractors + sections into Phenopackets
  genomics/         VCF scanning + File/Interpretation attachment (structural only)
  measurements/     TermBin, PercentileRange, EfwMeasurement dataclasses
```

`observer_phenopacket.py` and `viewpoint_phenopacket.py` duplicate the same seven private helpers (`_resolve_subject_ga`, `_biometry_feature`, `_narrative_feature`, `_dedup_by_hpo_id`, `_hpo_resource`, `_phenopacket_id`, `_subject_id`) by deliberate choice rather than sharing a utils module: at two files, that was ~3 duplicated lines per helper, not enough to justify the coordination cost of a shared module. A third builder, `gyn_phenopacket.py` (gynecologic exams), exists on a separate, not-yet-merged branch and follows the identical pattern - now that it would be three files sharing the same helpers, it's flagged as worth revisiting, and worth checking whether it has the same negation-handling gap fixed in Section 4, since it was copied from the same original code.

**Honestly still skeleton today** (each returns empty data, not silently wrong data):

| Module | Status |
|---|---|
| `etl/sections/maternal_history.py` | Skeleton |
| `etl/sections/placenta.py` | Skeleton |
| `etl/sections/amniotic_fluid.py` | Skeleton |
| `etl/sections/umbilical_cord.py` | Skeleton |
| `etl/sections/fetal_echo.py` | Skeleton (cardiac echo) |

ViewPoint HL7 parity work (multi-fetus biometry, real anatomy parsing, the `build_viewpoint_phenopacket` builder) landed as a separate PR stack this session - not re-derived here, since it has its own demo cell in `prenatalppkt.ipynb`.

## 9. Test suite tour

445 tests across 44 test files (plus doctests in `README.md` and select source modules, per `pytest.ini`'s `--doctest-modules --doctest-glob README.md`). Organized by what this walkthrough has already touched:

| Area | Test files |
|---|---|
| Growth-reference / TermBin mechanism | `test_mapping_loader.py`, `test_reference_range.py`, `test_term_bin.py`, `etl/test_term_bin_factory.py` |
| Extractors (Observer + ViewPoint HL7) | `etl/extractors/test_observer.py`, `etl/extractors/test_viewpoint_hl7.py` |
| Section parsers | `etl/sections/test_*.py` - one file per section (impression, anatomy, dating, EFW, ratios, indication) |
| fenominal / HPO | `hpo/test_fenominal_cr.py`, `hpo/test_hpo_parser.py` |
| Builders | `builders/test_observer_phenopacket.py`, `builders/test_viewpoint_phenopacket.py` |
| Genomics | `genomics/test_vcf.py`, `genomics/test_genomic.py` |
| Observer DTO / parser layer | `dto/observer/builders/*.py`, `parser/observer/fetuses/*.py` |
| Data dictionary tooling | `src/prenatalppkt/scripts/data_dict/tests/*.py` (co-located with the tool, not under `tests/`) |

**Testing philosophy:** real fixture data over mocks (`tests/data/*.json`, `*.txt`, `*.vcf` - not hand-mocked `prenatalppkt` internals), honest `skip` marks for genuinely unimplemented paths rather than tests that pass by asserting nothing, and parametrized sweeps across the real corpus for the data-dictionary tooling.

**This walkthrough's own contribution to that philosophy:** Sections 4 and 6 are a live example of what "real fixtures over mocks" is meant to catch - the fetal-anatomy path bug existed *because* its unit tests hand-rolled a JSON shape that agreed with the buggy code instead of with real data. Both bugs found while writing this notebook are now fixed with regression tests that were verified to fail before the fix and pass after, using real or realistically-shaped fixtures.